# A4. Multi-device networks and batching


Dendra networks can contain populations on different devices. Event and continuous connections are normally placed on the postsynaptic / synapse device, because delivery updates target synaptic state.

## Inspect placement

```python
net.devices()   # population name -> device
net.dtypes()    # population name -> dtype
```

A legacy `net.device()` call returns the first population's device and should not be used as a complete description of heterogeneous networks.

## Mixed CPU / accelerator placement

Within each timestep phase, Dendra queues accelerator-resident components before CPU components. This helps mixed CPU/GPU/MPS networks overlap accelerator kernels with host-side work when the backend permits it.

The dependency phases are still ordered:

```text
NetStim -> continuous target reset -> continuous NetCons -> event NetCons -> populations
```

Only independent components within a phase are reordered by device class.

Cross-device NetCons require moving presynaptic event/gate data to the postsynaptic/synapse device. This can introduce synchronization and copy overhead. Keep tightly coupled populations colocated unless there is a strong reason to split them.

## Batching networks

`Network.batch(n)` creates `n` independent replicas along a leading batch dimension:

```python
net = build_network()
net = net.batch(64)
net.initialize(dt)
net.run(tstop)
```

This is useful for parameter sweeps, Monte Carlo draws, and independent stimulus conditions. It is usually more efficient than building a Python list of independent networks.

## Slot-targeted batching

For banked point-process mechanisms, Dendra keeps two coordinate systems separate during batching:

```text
source_idx: population-flat source coordinate frame
target_idx: synapse-local slot coordinate frame
```

Users should select slots before connecting and let `Network.batch(n)` rebase the source and target coordinate systems.